In [1]:
import csv
import time
from pathlib import Path
from os.path import expanduser
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
from typing import Tuple, Optional
from datetime import datetime, timedelta


import lxml.html as lh
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------- Config ----------
GLOBAL_MAX_CONNECTIONS = 64          # connection pool size for requests
MAX_LOCATION_WORKERS = 16            # parallel fetches across stations
ROOT = Path(expanduser('~/phenology-urban/data/raw/CONUS/WU'))

In [2]:
DATE_PATTERNS = ['%m/%d/%Y', '%m/%d/%y', '%Y-%m-%d']

def _parse_date_guess(s: str):
    s = s.strip()
    for fmt in DATE_PATTERNS:
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    return None

def first_date_from_station_csv(root: Path, city: str, station: str):
    """Scan station.csv and return the earliest date (as date), or None."""
    path = root / city / f"{station}.csv"
    if not path.exists():
        return None
    first = None
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            ds = row.get('Date') or row.get('date') or row.get('DATE')
            if not ds:
                continue
            d = _parse_date_guess(ds)
            if d is None:
                continue
            if first is None or d < first:
                first = d
    return first

def make_session():
    s = requests.Session()
    retries = Retry(total=3, backoff_factor=0.3,
                    status_forcelist=(429, 500, 502, 503, 504))
    adapter = HTTPAdapter(max_retries=retries,
                          pool_connections=GLOBAL_MAX_CONNECTIONS,
                          pool_maxsize=GLOBAL_MAX_CONNECTIONS)
    s.mount('http://', adapter)
    s.mount('https://', adapter)
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (compatible; WeatherScraper/1.0; +https://example.com/bot)"
    })
    return s

SESSION = make_session()

# ---------- XPaths ----------
# Location text shown near the top of dashboard page
LOCATION_XPATH = '//*[@id="inner-content"]/div[1]/app-dashboard-header/div[2]/div/div[1]/span'
# A looser fallback in case markup shifts slightly
FALLBACK_LOCATION_XPATH = '//app-dashboard-header//span[1]'

def to_station_url(station: str, city: str, root: Path, offset_days: int = 1) -> str:
    """Daily graph URL for first CSV date (+offset_days), else fallback to dashboard."""
    d = first_date_from_station_csv(root, city, station)
    if d is None:
        return f"https://www.wunderground.com/dashboard/pws/{station}"
    d = d + timedelta(days=offset_days)  # e.g., 12/30/2024 -> 2024-12-31
    ds = d.strftime('%Y-%m-%d')
    return f"https://www.wunderground.com/dashboard/pws/{station}/graph/{ds}/{ds}/daily"

def extract_location_from_html(content: bytes) -> str | None:
    """Return clean location text or None if not found."""
    doc = lh.fromstring(content)
    els = doc.xpath(LOCATION_XPATH)
    if not els:
        els = doc.xpath(FALLBACK_LOCATION_XPATH)
    if not els:
        return None
    # Normalize whitespace
    return ' '.join(els[0].text_content().split())

# ---------- Location Parsing ----------
_LOC_RE = re.compile(r"""
    Elev(?:ation)?\s+
    (?P<elev_val>[-+]?\d+(?:\.\d+)?)      # elevation number
    \s*
    (?P<elev_unit>m|ft)                   # unit
    .*?                                   # anything (comma, spaces)
    (?P<lat_val>[-+]?\d+(?:\.\d+)?)       # latitude value
    \s*°?\s*
    (?P<lat_hem>[NS])                     # N/S
    \s*,\s*
    (?P<lon_val>[-+]?\d+(?:\.\d+)?)       # longitude value
    \s*°?\s*
    (?P<lon_hem>[EW])                     # E/W
""", re.IGNORECASE | re.VERBOSE)

def _to_signed(value: float, hemi: str) -> float:
    hemi = hemi.upper()
    if hemi in ("S", "W"):
        return -abs(value)
    return abs(value)

def parse_location_text(text: str) -> Tuple[Optional[float], Optional[float], Optional[float]]:
    """
    Returns (ele_m, lon_dd, lat_dd) or (None, None, None) if not parseable.
    - ele_m: elevation in meters (float)
    - lon_dd: longitude in decimal degrees (E positive, W negative)
    - lat_dd: latitude in decimal degrees (N positive, S negative)
    """
    if not text:
        return (None, None, None)

    m = _LOC_RE.search(text)
    if m:
        elev_val = float(m.group("elev_val"))
        elev_unit = m.group("elev_unit").lower()
        ele_m = elev_val if elev_unit == "m" else elev_val * 0.3048

        lat = _to_signed(float(m.group("lat_val")), m.group("lat_hem"))
        lon = _to_signed(float(m.group("lon_val")), m.group("lon_hem"))
        return (round(ele_m, 3), round(lon, 6), round(lat, 6))

    return (None, None, None)

# ---------- Fetch Station Location ----------
def fetch_station_location(city: str, station: str, timeout: int = 10):
    """Fetch location for one station using the daily graph URL for first CSV date."""
    url = to_station_url(station, city, ROOT)
    try:
        resp = SESSION.get(url, timeout=timeout)
        if resp.status_code >= 500:
            time.sleep(0.3)
            resp = SESSION.get(url, timeout=timeout)
        if resp.status_code == 404:
            return (station, url, None, None, None, None)
        loc = extract_location_from_html(resp.content)
        ele_m, lon_dd, lat_dd = parse_location_text(loc)
        return (station, url, loc, ele_m, lon_dd, lat_dd)
    except Exception:
        return (station, url, None, None, None, None)

# ---------- Process Stations ----------
def get_city_to_stations(root: Path) -> dict[str, list[str]]:
    """Group station names (stems) by city folder; deduplicate within each city."""
    city_map: dict[str, set[str]] = {}
    for p in root.glob('*/*.csv'):
        city = p.parent.name
        station = p.stem
        city_map.setdefault(city, set()).add(station)
    # convert sets to sorted lists
    return {city: sorted(stations) for city, stations in city_map.items()}

def write_city_locations(city: str, rows: list[dict], root: Path):
    out_path = root / city / 'location.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['station', 'url', 'location', 'elevation', 'lon', 'lat'])
        writer.writeheader()
        writer.writerows(rows)
    print(f"[{city}] wrote {out_path} ({len(rows)} stations)")

def build_locations_for_city(city: str, stations: list[str]):
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_LOCATION_WORKERS) as ex:
        futs = [ex.submit(fetch_station_location, city, s) for s in stations]
        for fut in as_completed(futs):
            station, url, loc, elev, lon, lat = fut.result()
            rows.append({
                'station': station,
                'url': url,
                'location': loc or '',
                'elevation': elev,
                'lon': lon,
                'lat': lat
            })
    return rows

In [ ]:
def main():
    city_to_stations = get_city_to_stations(ROOT)
    if not city_to_stations:
        print(f"No stations found under {ROOT}")
        return

    for city, stations in sorted(city_to_stations.items()):
        rows = build_locations_for_city(city, stations)
        write_city_locations(city, rows, ROOT)

if __name__ == "__main__":
    main()